In [ ]:
###Select dataset to analyze
path = 'Z:/Augusto/Images/SoRa/03042024-SoRa-Tianhong' #avoid writing a dash / at the end
#path = 'D:/Tianhong/Images/24012024-SoRa' #Tianhong Wang's path

genotype = 'Nup98-KDM5A-mEGFP-SoRa'
dataset_number = 63

#Define which channel corresponds to fluorescent protein and which one corresponds to nucleus marker ('fp' or 'nucleus')
ch1 = 'nucleus'
ch2 = 'fp'

print("Analysis running")

In [ ]:
###Define Parameters for segmentation 
#A file will be generated with this information
#Notes:
#low_sigma
#1.5 low sigma excludes noise and it is a good compromise for nup98-hoxa9 and nup98-kdm5a (mEGFP and mCherry-tagged);
#2.5 low sigma for Nup98-KDM5Amut
#high_sigma
#4.5 high sigma seems to be a good compromise for nup98-hoxa9 and nup98-kdm5a (mEGFP and mCherry-tagged);
#also used for Nup98-KDM5Amut
#gaussian_threshold
#1e-4 is a threshold optimized after inspection of various datasets nup98-hoxa9 and nup98-kdm5a (mEGFP and mCherry-tagged);
#5e-4 for Nup98-KDM5Amut.
#min_size
#4 Minimum size to exclude speckles (4)
#erosion_size
#5 works well. 

#Relevant values for analysus
segmentation_method = "DoG and thresholding"
low_sigma = 1.5 #DoG low sigma 
high_sigma = 4.5 #DoG high sigma 
gaussian_threshold = 1e-4 #Threshold optimized after inspection of various datasets (5e-4 for KDM5A mut)
min_size = 4 #Minimum size to exclude speckles (4)

erosion_size = 5 #Adjust the size of eroded element (5 layers of pixels around detected condensates)
#Write comments that may be relevant for the analysis
Comments = 'C-terminus mEGFP-tagged Nup98-KDM5A condensates'


In [ ]:
# #####
#####Define parameters for cellpose
#####Cell detection

model_type = 'cyto' #nuclei', #'cyto', # cyto2 #models provided by cellpose
estimated_diameter = 750 #value 750 covers entire large cells SoRa; #250 covers small cells confocal; modify if necessary
flow_threshold = 0.7 #flow_threshold 0.7. Higher threshold enables more permissive cell detection
cellprob_threshold = 0.35 #0.35 works well 
contrast_stretching = 0 #takes percentiles on both edges of histogram (eg., for 5, 5th and 95th percentiles) values 0-100

In [ ]:
#####Run basic functions (readDataset, plot_image, implement_cellPose)
#####
#This function reads ome tif files generated by micro manager.
#It takes the path to your files, the genotype (as my data is organized in folders named after the genotype of the imaged cells)
#and dataset number to select one of the datasets in the genotype folder
def readDataset(path, genotype_number):
    
    datafolder = os.path.join(path, genotype_number)
    
    os.chdir(datafolder)
    print(os.getcwd())
    
    DatasetsinFolder = os.listdir()
    print(DatasetsinFolder)
    
    for element in DatasetsinFolder:
        if ".ome.tif" in element:
            Dataset = element
            print("input file:", Dataset)
            break
    
    image_stack = io.imread(Dataset, plugin='pil') #We need to read the .ome files with 'pil' plugin due to bugs recognizing the shape of the files in other formats
    #print(image_stack.shape) #the shape of these ome.tif files should be 2 channels, 54 stacks, and 2048 times 2048 images

    return image_stack

#####
#####
#Daniel Foyt wrote this code to plot numpy arrays
def plot_image(image,title = '',size = 20,cmap='gray',cmapp_bar = True):
    fig, ax = plt.subplots(figsize=(size,size))
    plt.imshow(image, cmap=cmap)
    if cmapp_bar == True:
        cbar = plt.colorbar()
    ax.set_title(title)
    plt.show()
    
######
######
#set up cell_pose
#import cellpose
#from cellpose import models
##Cellpose was developed at janelia farm by Carsen Stringer and Marius Pachitariu
##https://doi.org/10.1101/2022.04.01.486764
#import torch
##t = torch.cuda.get_device_properties(0).total_memory
##r = torch.cuda.memory_reserved(0)
##a = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 8000000000 else "cpu")
#This code implements models for cell pose. This function allows to call cell pose later in the code. 
#It takes the model type used for the prediction of cell pose masks and an array (image) where the masks will be applied.
def implement_cellPose(model_type, estimated_diameter, flow_threshold, cellprob_threshold, image_slice):

    device = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 8000000000 else "cpu")
    device_type = device.type
    gpu = True if "cuda" in str(device_type) else False

    # https://github.com/MouseLand/cellpose/blob/main/cellpose/models.py
    model = models.Cellpose(
                model_type = model_type,
                device = device,
                gpu = gpu)

    estimated_diameter = estimated_diameter #estimated size of cells (pixels)

    # picked some image to test on 
    #sum_image_smoth = image[18][1]
    #sum_image_smoth2 = io.imread('D:/Augusto/Images/ConfocalImaging/20221115/mCherry-Nup98nt1833-KDM5Ant4666-spy650-640nm30_1/mCherry-Nup98nt1833-KDM5Ant4666-spy650-640nm30_1_MMStack_5-Pos000_000.ome.tif')[1][25] 

    sum_image_smoth = image_slice
    batch = [sum_image_smoth]

    #flow_threshold 0.75
    masks, flows, styles, diams = model.eval(batch,
                                                 diameter = estimated_diameter,
                                                 flow_threshold = flow_threshold,
                                                 cellprob_threshold = cellprob_threshold,
                                                 channels = None,
                                                 tile = False)
    
    return masks, flows, styles, diams, batch


In [ ]:
#####Select folder, dataset, and channels
#######
#######
#This code implements functions for reading tif.ome files

import os
import re
import skimage 
from skimage import io
import matplotlib.pyplot as plt
import numpy as np
import tifffile as tf

#print(skimage.__version__)
#skimage reference
#Stéfan van der Walt, Johannes L. Schönberger, Juan Nunez-Iglesias, François Boulogne, Joshua D. Warner, Neil Yager, Emmanuelle Gouillart, Tony Yu and the scikit-image contributors. scikit-image: Image processing in Python. PeerJ 2:e453 (2014) https://doi.org/10.7717/peerj.453
#https://biomedicalhub.github.io/python-data/skimage.html - Read about image processing skimage

#path = 'D:/Augusto/Images/SoRa/16112023-SoRa' #avoid writing a dash / at the end
dataset_date = path.split('/')[-1]
#genotype = 'mEGFP-Nup98-HoxA9-SoRa'
#dataset_number = 14
genotype_number = genotype + '_' + str(dataset_number)

#Read the dataset
image_stack = readDataset(path, genotype_number)
image_shape = np.shape(image_stack) 

#Create annotation file
output_folder =  path + "/" + "analysis_" + dataset_date + '_' + genotype
filename_condensates = output_folder + "/" + dataset_date + '_' + genotype_number + '_condensates.txt'
filename_cells = output_folder + "/" + dataset_date + '_' + genotype_number + '_cells.txt'
filename_parameters = output_folder + "/" + dataset_date + '_' + genotype_number + '_parameters.txt'
print("output folder:", output_folder)
print("output files:", filename_condensates, "\n", filename_cells, "\n", filename_parameters, "\n")
#The resulting shape of the dataset has the two channels stacked on each other. 
no_of_channels = len(image_shape) - 1 #Retrieving the number of channels in our dataset.

#Thus, the stack is divided by the number of channels, and each slice assigned to a channel.
stacks_perChannel = int(np.shape(image_stack)[0] / no_of_channels)

print("stacks per channel: ", stacks_perChannel)
print("shape of dataset: ", (np.shape(image_stack))) #shape of the read dataset using pil (stacks X channels, y pixels, x pixels)

#Setting the last slices for each channel (due to python 0-indexing)
last_image_ch1_slice = stacks_perChannel
last_image_ch2_slice = stacks_perChannel * 2

# print("last_image_fp_slice", last_image_fp_slice)
# print("last_image_spy650_slice", last_image_spy650_slice)

image_ch1 = image_stack[0:last_image_ch1_slice, :, :] 
image_ch2 = image_stack[stacks_perChannel:last_image_ch2_slice, :, :]
#plt.imshow(image_stack[0, :, :])

#Assign proper channels to corresponding variables for further analysis
if ch1 == 'fp':
    image_chProtein = image_ch1
elif ch1 == 'nucleus':
    image_chNuclei = image_ch1
else:
    raise ValueError("ch1 must be either 'fp' or 'nucleus'")

if ch2 == 'nucleus':
    image_chNuclei = image_ch2
elif ch2 == 'fp':
    image_chProtein = image_ch2
else:
    raise ValueError("ch2 must be either 'fp' or 'nucleus'")
    


In [ ]:
### ####Choose slice to analyze

#Iterate over all cells
print("Size of stacks per channel (should be the same)")
print("Protein channel", np.shape(image_chProtein))
print("Chromatin channel", np.shape(image_chNuclei))

#Set contrast
# vmin=100
# vmax=300
vmin = np.percentile(image_chProtein, 20)
vmax = np.percentile(image_chProtein, 99.8)

for index, Zslice in enumerate(image_chProtein):
    
    print("Slice ", index)
    plt.figure(figsize=(10, 10))
    plt.imshow(Zslice, cmap='gray', vmin=vmin, vmax=vmax)
    plt.show()
    # Close the figure explicitly to free up memory
    plt.close()
    
define_slice = "Choose a slice for analysis\n"
analysis_slice = int(input(define_slice))


###Images (slices) for further analysis
image_chProtein_selected = image_chProtein[analysis_slice]
image_chNuclei_selected = image_chNuclei[analysis_slice]

####
####Print the slice chosen for analysis

print("This is the slice that you chose\nslice: ", analysis_slice)

# Create a subplot with 1 row and 2 columns
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot protein channel
axes[0].imshow(image_chProtein_selected, cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('Protein channel')

# Plot chromatin channel
axes[1].imshow(image_chNuclei_selected, cmap='gray')
axes[1].set_title('Chromatin channel')

# Adjust layout for better spacing
plt.tight_layout()

# Show the plot
plt.show()




In [ ]:
#DEFINE SLICE FOR SPEED
###Images (slices) for further analysis
# image_chProtein_selected = image_chProtein[10]
# image_chNuclei_selected = image_chNuclei[10]

In [ ]:
#####Call cell pose on slices in a z-stack and select a slice
import cellpose
from cellpose import models
from cellpose import utils
from skimage import exposure
#Cellpose was developed at janelia farm by Carsen Stringer and Marius Pachitariu
#https://doi.org/10.1101/2022.04.01.486764

import torch
#t = torch.cuda.get_device_properties(0).total_memory
#r = torch.cuda.memory_reserved(0)
#a = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 8000000000 else "cpu")

print((np.shape(image_chNuclei_selected)))

#Contrast enhancing for spy650 channel
p_value_low = contrast_stretching
p_value_high = 100 - contrast_stretching
p_low, p_high = np.percentile(image_chNuclei_selected, (p_value_low, p_value_high))
#img_rescaled = exposure.rescale_intensity(image_chNuclei_selected, in_range=(p_low, p_high))
img_rescaled = image_chNuclei_selected

masks, flows, styles, diams, batch = implement_cellPose(model_type, estimated_diameter, flow_threshold, cellprob_threshold, img_rescaled)
#print(masks)

plot_image(batch[0],title = 'raw',size = 20,cmap='gray',cmapp_bar = False)
plot_image(masks[0]*100,title = 'masks',size = 20,cmap='cool',cmapp_bar = False)
outline_masks = cellpose.utils.masks_to_outlines(masks[0])
plot_image(outline_masks, title = 'masks',size = 20,cmap='cool',cmapp_bar = False)

print('Number of cells masked:', len(np.unique(masks[0]))-1) 
    # perimeter_masks = cellpose.utils.get_mask_perimeters(masks[0])
    # perimeter_masks
    
selected_slice_mask = masks[0]

# print('outline_masks', np.size(outline_masks))
# print(outline_masks)

    

In [ ]:
#Segments condensates using the following skimage pipeline:
#1 - Calculates DoG image
#2 - Threshold for each cell in DoG image
#3 - Uses a threshold to generate a binary images
#4 - Put together binary images in a single matrix
#5 - Area closing of hollow detected condensates
#6 - Labels masks (according to contiguous pixels)
#https://haesleinhuepf.github.io/BioImageAnalysisNotebooks/20_image_segmentation/09a_gauss_otsu_labeling.html

from skimage import io, color, filters, measure
from skimage.filters import threshold_otsu
from skimage.filters import difference_of_gaussians
from skimage.segmentation import relabel_sequential
from skimage import measure
from skimage import morphology
import pyclesperanto_prototype as cle

#Plot original image to segment
plt.imshow(image_chProtein_selected, cmap='viridis', interpolation='nearest')
plt.title('Original image')
plt.colorbar()
plt.show()

#Convert original image into a DoG
image_chProtein_selected_DoG = skimage.filters.difference_of_gaussians(image_chProtein_selected, low_sigma=low_sigma, high_sigma=high_sigma, mode='constant', cval=0.0, channel_axis=None, truncate=4.0) #truncate detection at 4 sds is default
plt.imshow(image_chProtein_selected_DoG)
plt.suptitle('DoG image\nHigh sigma = ' + str(high_sigma) + ' - Low sigma = ' + str(low_sigma)) 
#plt.subtitle('high sigma = ' + str(high_sigma) + '/ low sigma = ' + str(low_sigma) )
plt.show()

#Threshhold DoG image
image_chProtein_selected_DoG_threshold = image_chProtein_selected_DoG > gaussian_threshold

#Close hollow condensate masks
image_chProtein_selected_DoG_threshold_closed = skimage.morphology.area_closing(image_chProtein_selected_DoG_threshold, area_threshold=20000, 
                                                                   connectivity=10, parent=None, tree_traverser=None)
#Label individual condensate masks
image_chProtein_selected_DoG_threshold_closed_labeled = skimage.morphology.label(image_chProtein_selected_DoG_threshold_closed, background=None, return_num=False, connectivity=None)
plt.imshow(image_chProtein_selected_DoG_threshold_closed_labeled)
print('condensate masks (w/ small objects):', len(np.unique(image_chProtein_selected_DoG_threshold_closed_labeled)))
plt.title('Segmented and labeled condensate masks (w/ small objects)')
plt.show() 

#Remove small objects
image_chProtein_selected_DoG_threshold_closed_labeled_rmsmall = skimage.morphology.remove_small_objects(image_chProtein_selected_DoG_threshold_closed_labeled, min_size=min_size, connectivity=None, out=None)
plt.imshow(image_chProtein_selected_DoG_threshold_closed_labeled_rmsmall[300:1800, 300:1800], interpolation='nearest')
print('condensate masks:', len(np.unique(image_chProtein_selected_DoG_threshold_closed_labeled_rmsmall)))
plt.title('Segmented and labeled condensate masks (wo/ small objects)')
plt.show()      

#Re-label condensate masks sequencially, starting from 1 (0 is background)
# print('condensate masks:', np.unique(seg_lab_cdmasks))
image_chProtein_segmented, fw, inv  = skimage.segmentation.relabel_sequential(image_chProtein_selected_DoG_threshold_closed_labeled_rmsmall, offset=1)
#print('condensate masks:', len(np.unique(seg_lab_cdmasks)))
# print('fw', fw)
# print('inv', inv)

###
###Plot individual masks to assess segmentation
###

###Select unique masks
unique_masks = np.unique(selected_slice_mask) 
print("unique cell masks:", unique_masks, " - (0 is background)")

#Iterate over all masks (cells) and plot them
for mask_number in unique_masks:
    
    binary_cell_mask = selected_slice_mask == mask_number #make binary mask of cell
    
    #Find mask coordinates
    mask_cell_rows, mask_cell_cols = np.where(binary_cell_mask == 1)
    y_low = np.min(mask_cell_rows) #Coordinates of area around mask
    y_up = np.max(mask_cell_rows)
    x_low = np.min(mask_cell_cols)
    x_up = np.max(mask_cell_cols)
    
    #Plot segmented mask
    image_chProtein_segmented_plotting = image_chProtein_segmented[y_low:y_up,x_low:x_up]
    image_chProtein_selected_plotting = image_chProtein_selected[y_low:y_up,x_low:x_up]
    contour_mask_plotting = np.argwhere(outline_masks[y_low:y_up,x_low:x_up])
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    axes[0].imshow(image_chProtein_segmented_plotting)
    axes[0].scatter(contour_mask_plotting[:, 1], contour_mask_plotting[:, 0], color='white', marker='o', s=5)  # Adjust marker style and size
    axes[0].set_title('Segmented mask ' + str(mask_number))
      
    vmin = np.percentile(image_chProtein_selected_plotting, 20)
    vmax = np.percentile(image_chProtein_selected_plotting, 99)
    axes[1].imshow(image_chProtein_selected_plotting, cmap='gray', vmin = vmin, vmax = vmax)
    axes[1].set_title('Protein channel')
    
    # Adjust layout for better spacing
    plt.tight_layout()

    # Show the plot
    plt.show()

In [ ]:
# Obtain data points for the masked condensates 
from scipy.ndimage import center_of_mass
from skimage import measure
import pandas as pd

maskID_condensates = []
centroids_x_condensates = []
centroids_y_condensates = []
area_pixels_condensates = []
raw_mass_condensates = []
eccentricities_condensates = []
for mask in np.unique(image_chProtein_segmented):
    if mask == 0:
        continue
        
    binary_mask = image_chProtein_segmented == mask #make binary mask

    center = center_of_mass(binary_mask) #Calculate the center of mass
    center_y, center_x = center #Print the center coordinates
    center_x = round(center_x)
    center_y = round(center_y)
    
    area = np.sum(binary_mask) #Calculate area
    
    mask_rows, mask_cols = np.where(binary_mask == 1) #find pixels that correspond to mask
    raw_mass_cd = np.sum(image_chProtein_selected[mask_rows, mask_cols]) #Calculate mass
    
    props_cd = measure.regionprops(binary_mask.astype(int)) #Measure properties of the mask and extract eccentricity 
    eccentricity_cd = props_cd[0].eccentricity
    
    maskID_condensates.append(mask - 1) #substract one to make id coincide with index
    centroids_x_condensates.append(center_x)
    centroids_y_condensates.append(center_y)
    area_pixels_condensates.append(area)
    raw_mass_condensates.append(raw_mass_cd)
    eccentricities_condensates.append(eccentricity_cd)
    
#     print('mask:', mask)
#     print('mask x:', center_x)
#     print('mask y:', center_y)
#     print('area:', area)

# Create a DataFrame from the lists
df_condensates = pd.DataFrame({'condensate_id': maskID_condensates,
                    'y': centroids_y_condensates, 'x': centroids_x_condensates, 
                    'raw_mass': raw_mass_condensates, 'size': area_pixels_condensates,
                    'ecc': eccentricities_condensates})

print(df_condensates)



# Plot the original fp image
vmin = np.percentile(image_chProtein_selected, 20) #set contrast
vmax = np.percentile(image_chProtein_selected, 99.8) #set contrast
plt.imshow(image_chProtein_selected, origin='upper', vmin = vmin, vmax = vmax)
#plt.imshow(image_chProtein_selected[500:1500,500:1500], origin='upper')
plt.title("Original fp image")
plt.show()

# Plot the spots on top of the image colorcoded by area
plt.imshow(image_chProtein_selected, origin='upper')
plt.scatter(centroids_x_condensates, centroids_y_condensates, c = area_pixels_condensates, cmap='cool', marker='.', linestyle='None')
plt.colorbar()
plt.gca().invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
plt.xlim(100, 1900)
plt.ylim(1900, 100)
plt.title("Area")
plt.show()

# Plot the spots on top of the image colorcoded by mean intensity
plt.imshow(image_chProtein_selected, origin='upper')

mean_condensates = [rm_cd / a_cd for rm_cd, a_cd in zip(raw_mass_condensates, area_pixels_condensates)]
plt.scatter(centroids_x_condensates, centroids_y_condensates, c = mean_condensates, cmap='cool', marker='.', linestyle='None')
plt.colorbar()
plt.gca().invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
plt.xlim(100, 1900)
plt.ylim(1900, 100)
plt.title("Mean Intensity")
plt.show()

# Plot the spots on top of the image colorcoded by mean intensity
plt.imshow(image_chProtein_selected, origin='upper')
plt.scatter(centroids_x_condensates, centroids_y_condensates, c = raw_mass_condensates, cmap='cool', marker='.', linestyle='None')
plt.colorbar()
plt.gca().invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
plt.xlim(100, 1900)
plt.ylim(1900, 100)
plt.title("Raw Mass")
plt.show()

# Plot the masks
print("Segmented and labeled masks")
#cle.imshow(seg_lab_cdmasks[500:1500,500:1500], labels=True)
cle.imshow(image_chProtein_segmented, labels=True)




In [ ]:
#Plotting histogram of condensate sizes

#Filter values under 1000
filtered_area_pixels_condensates = [x for x in area_pixels_condensates if x < 1000]

#Plot histogram
plt.hist(filtered_area_pixels_condensates, bins=1000, color='blue', edgecolor='black')
plt.suptitle('Histogram of condensates area\nHigh sigma = ' + str(high_sigma) + ' - Low sigma = ' + str(low_sigma) + ' - min condensate size = ' + str(min_size) + ' pix')
plt.xlabel('Area of condensates (pixels)')
plt.ylabel('Frequency')

#Set the x-axis and y-axis limits
plt.xlim(0, 1000)  # set the x-axis limits from 0 to 6

#Plot mean
filtered_area_pixels_condensates_mean = np.mean(filtered_area_pixels_condensates)
#filtered_area_pixels_condensates_sd = np.std(filtered_area_pixels_condensates)
#percentile_95 = np.percentile(filtered_area_pixels_condensates, 95)
#lower_bound = filtered_area_pixels_condensates_mean - 1 * filtered_area_pixels_condensates_sd
#upper_bound = filtered_area_pixels_condensates_mean + 1 * filtered_area_pixels_condensates_sd

plt.axvline(filtered_area_pixels_condensates_mean, color='red', linestyle='dashed', linewidth=2, label=f'filtered_area_pixels_condensates_mean: {filtered_area_pixels_condensates_mean:.2f}')

#plt.axvline(lower_bound, color='green', linestyle='dashed', linewidth=2, label=f'Lower Bound: {lower_bound:.2f}')
#plt.axvline(upper_bound, color='orange', linestyle='dashed', linewidth=2, label=f'Upper Bound: {upper_bound:.2f}')
#plt.axvline(percentile_95, color='cyan', linestyle='dashed', linewidth=2, label=f'percentile_95: {percentile_95:.2f}')

plt.show()

In [ ]:
#####Match cellPose and trackpy results and generate .txt output
#This chunk of code assings condensates to masks (cells)
#Initialize a condensates_stats np array to fill with various condensate values.
from skimage import measure
from skimage import morphology
                                                  
#Assign large condensates to a cell mask

# maskID_condensates.append(mask-1) #substract one to make id coincide with index
# centroids_x_condensates.append(center_x)
# centroids_y_condensates.append(center_y)
# raw_mass_condensates.append(raw_mass_cd)
# area_pixels_condensates.append(area)
# eccentricities_condensates.append(eccentricity_cd)

condensate_stats = np.empty((0, 7), float)
for cd_index in df_condensates.index:
    
    cd_id = cd_index
    cd_xpixel = round(df_condensates.iloc[cd_index]['x']) #condensate center in x
    cd_ypixel = round(df_condensates.iloc[cd_index]['y']) #condensate center in y 
    cd_raw_mass = df_condensates.iloc[cd_index]['raw_mass'] #raw mass 
    cd_size = df_condensates.iloc[cd_index]['size'] #area
    cd_ecc = df_condensates.iloc[cd_index]['ecc'] #eccentricity
    
    condensates_in_cell_mask = selected_slice_mask[cd_ypixel, cd_xpixel] #Finding the mask that corresponds to condensate centroid location
    
    #Append a new row to spot_stats for each spot
    ######This may be modified to accomodate the outcome agreed in a lab meeting#########
    condensate_stats = np.append(condensate_stats, np.array([[condensates_in_cell_mask, 
        cd_id, cd_xpixel, cd_ypixel, cd_raw_mass, cd_size, cd_ecc]]), axis=0)
    
#Plot fp and spy650 channels for reference
vmin = np.percentile(image_chProtein_selected, 20) #set contrast
vmax = np.percentile(image_chProtein_selected, 99.8) #set contrast
plt.imshow(image_chProtein_selected, vmin = vmin, vmax = vmax)
plt.show()
plt.imshow(image_chNuclei_selected)
plt.show()

#print("condensate_stats", condensate_stats)
#find the unique masks in the array (number of masks) and iterate through them
unique_masks = np.unique(selected_slice_mask) 
print("unique masks:", unique_masks)

selected_cells_stats = np.empty((0, 8), float)
selected_nucleoplasms = []
selected_coordinates = []
for mask_number in unique_masks:
    
    #####
    binary_cell_mask = selected_slice_mask == mask_number #make binary mask

    #Find mask coordinates
    mask_cell_rows, mask_cell_cols = np.where(binary_cell_mask == 1)
    y_low = np.min(mask_cell_rows) #Coordinates of area around mask
    y_up = np.max(mask_cell_rows)
    x_low = np.min(mask_cell_cols)
    x_up = np.max(mask_cell_cols)
    
    #Find center of mass for mask (center of cell)
    center_cell = center_of_mass(binary_cell_mask) #Calculate the center of mass of cell mask
    center_cell_y, center_cell_x = center_cell #Print the center coordinates
    center_cell_x = round(center_cell_x)
    center_cell_y = round(center_cell_y)
    
    #####
    condensates_in_mask_index = condensate_stats[:,0] == mask_number #index of condensates assigned to current mask
    #print("condensates_in_mask_index", condensates_in_mask_index)
    condensates_in_mask_x = condensate_stats[condensates_in_mask_index,2] #obtain the x position of condensates in current mask 
    condensates_in_mask_y = condensate_stats[condensates_in_mask_index,3] #obtain the y position of condensates in current mask
    
    mask_cell_rows, mask_cell_cols = np.where(binary_cell_mask == 1) #Find where mask is located
    
    if mask_number == 0:
        background = round(np.mean(image_chProtein_selected[mask_cell_rows, mask_cell_cols])) #1st mask is background 
    
    #Obtain stats for cell mask
    #mean_expression = round(np.mean(image_chProtein_selected[mask_cell_rows, mask_cell_cols]))
    raw_cell_expression = np.sum(image_chProtein_selected[mask_cell_rows, mask_cell_cols])
    cell_area = np.sum(binary_cell_mask)
    
    #Obtain nucleoplasm mask
    binary_no_fp_mask = image_chProtein_segmented == 0
    binary_nucleoplasm_mask = binary_no_fp_mask & binary_cell_mask
    
    #Erode nucleoplasm mask
    #erosion_size is defined above
    for iterations in range(erosion_size):
        binary_nucleoplasm_mask = morphology.binary_erosion(binary_nucleoplasm_mask)

    nucleoplasm_mask_rows, nucleoplasm_mask_cols = np.where(binary_nucleoplasm_mask == 1) #Find where nucleoplasm mask for current cell is located
    #Obtain stats for nucleoplasm mask
    raw_nucleoplasm_expression = np.sum(image_chProtein_selected[nucleoplasm_mask_rows, nucleoplasm_mask_cols])
    nucleoplasm_area = np.sum(binary_nucleoplasm_mask)
    
    ####Plot various stats of the cell mask currently analyzed
    
    #Plot nucleoplasm
    plt.imshow(binary_nucleoplasm_mask[y_low:y_up,x_low:x_up])
    plt.title("nucleoplasm mask " + str(mask_number) + "\nfp")
    plt.show()
        
    #Create a figure with two subplots, for each of the segmented channels
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    contours_cell_mask = measure.find_contours(binary_cell_mask, 0.5)[0] #this line returns only the first contour detected
    vmin = np.percentile(image_chProtein_selected, 20) #set contrast
    vmax = np.percentile(image_chProtein_selected, 99.8) #set contrast
    ax1.imshow(image_chProtein_selected, origin='upper', vmin = vmin, vmax = vmax)
    ax1.plot(contours_cell_mask[:, 1], contours_cell_mask[:, 0], 'w', linewidth=1) #plot contours
    ax1.scatter(condensates_in_mask_x, condensates_in_mask_y, marker='o', color = "r", linewidth=2, alpha = 0.4)
    ax1.scatter(center_cell_x, center_cell_y, marker='*', color = "w", linestyle='None')
    ax1.set_xlim(x_low, x_up) #zoom to mask currently analyzed
    ax1.set_ylim(y_up, y_low)
    ax1.invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
    ax1.set_title("mask " + str(mask_number) + "\nfp")
    ax1.set_ylim(ax1.get_ylim()[::-1])  # Reverse the y-axis limits

    ax2.imshow(image_chNuclei_selected, origin='upper', cmap='viridis')
        #,vmin = background, vmax = 3 * (background))
    ax2.plot(contours_cell_mask[:, 1], contours_cell_mask[:, 0], 'r', linewidth=1) #plot contours
    ax2.scatter(center_cell_x, center_cell_y, marker='*', color = "w", linestyle='None')
    ax2.set_xlim(x_low, x_up) #zoom to mask currently analyzed
    ax2.set_ylim(y_up, y_low)
    ax2.invert_yaxis() # Invert the y-axis and set the x-axis and y-axis limits for the zoomed region
    ax2.set_title("mask " + str(mask_number) + "\nspy650")
    ax2.set_yticklabels([])  # Remove y-axis tick labels
    ax2.set_ylim(ax2.get_ylim()[::-1])  # Reverse the y-axis limits
    
    plt.show()
    
    #plot different features of condensates associated with the mask currently analyzed
    print('mask id:', mask_number)
    mean_expression = round(raw_cell_expression / cell_area)
    print('mean mask intensity (raw)', mean_expression)
    print('background:', background)
    average_nucleoplasm_intensity =  raw_nucleoplasm_expression / nucleoplasm_area
    print('average nucleoplasm intensity (raw)', average_nucleoplasm_intensity)
    number_of_condensates = np.sum(condensates_in_mask_index)
    print('number of condensates', number_of_condensates)
    average_raw_mass_of_condensates = np.mean(condensate_stats[condensates_in_mask_index, 4])
    print('average raw mass of condensates', average_raw_mass_of_condensates)
    average_condensate_size = np.mean(condensate_stats[condensates_in_mask_index, 5])
    print('average condensate size', average_condensate_size)
      
    if mask_number == 0:
        #Save nucleoplasm mask
        binary_no_cell_masks = binary_nucleoplasm_mask
    elif mask_number > 0:
        #Ask if cell is good for further analysis
        ask_analyzed_cell = "Is this cell (mask) " + str(mask_number) + " good for further analysis (y/n)\n"
        answer_analyzed_cell = input(ask_analyzed_cell)
        
        if answer_analyzed_cell == 'y':
            
            selected_cells_stats = np.append(selected_cells_stats, np.array([[mask_number, 
                center_cell_x, center_cell_y, raw_cell_expression, cell_area, background, 
                raw_nucleoplasm_expression, nucleoplasm_area]]), axis=0)
            
            #If selected, save nucleoplasm image for further plotting
            selected_nucleoplasms.append(binary_nucleoplasm_mask[y_low:y_up,x_low:x_up])
            selected_coordinates.append([y_low,y_up,x_low,x_up])

print("\nAnalysis finished")

In [ ]:
###Save images of the cells selected
import matplotlib.colors
from matplotlib.patches import Rectangle
import io
import shutil

###This piece of code was taken from cle.imshow source code. It generates
###random colors for labels.
###Reference:
###https://github.com/clEsperanto/pyclesperanto_prototype/blob/571401a0ad215397423eda6833cd177e1086069b/pyclesperanto_prototype/_tier9/_imshow.py
from matplotlib.colors import ListedColormap
from numpy.random import MT19937
from numpy.random import RandomState, SeedSequence
rs = RandomState(MT19937(SeedSequence(3)))
lut = rs.rand(65537, 3)
lut[0, :] = 0
# these are the first four colours from matplotlib's default
lut[1] = [0.12156862745098039, 0.4666666666666667, 0.7058823529411765]
lut[2] = [1.0, 0.4980392156862745, 0.054901960784313725]
lut[3] = [0.17254901960784313, 0.6274509803921569, 0.17254901960784313]
lut[4] = [0.8392156862745098, 0.15294117647058825, 0.1568627450980392]
cmap  = matplotlib.colors.ListedColormap(lut)

####Create folders for analysis files
###
# Specify the folder path
if not os.path.exists(output_folder): # Check if the folder exists
    #Create the folder
    os.makedirs(output_folder)
    print("Folder created successfully!")
else:
    print("Folder already exists.")
    
#Create folder for all images
output_folder_images = output_folder + "/Images"
print(output_folder_images)

if not os.path.exists(output_folder_images): # Check if the folder exists
    os.makedirs(output_folder_images) #Create the folder
    print("Folder created successfully!")
else:
    print("Folder already exists.")

#If folder for images in dataset exists, delete and create new one
output_folder_images_dataset = output_folder + "/Images" + "/Images_" + str(dataset_date) + "-" + str(genotype_number)
print(output_folder_images_dataset)

if os.path.exists(output_folder_images_dataset): # Check if the folder exist
    shutil.rmtree(output_folder_images_dataset) #Delete the folder
    print("Folder already exists, folder deleted.")
    os.makedirs(output_folder_images_dataset) #Create the folder
    print("Folder created successfully!")
else:
    os.makedirs(output_folder_images_dataset)
    print("Folder created successfully!") #Create the folder

###Plot whole slice images
###
# Set the color for the contours overlay on nuclear image
contours_value = 64000 # white color
# Create a copy of the original image to overlay the contours
contours_image_chNuclei_selected = np.copy(image_chNuclei_selected) 
# Apply the contours mask to the overlay image
contours_image_chNuclei_selected[outline_masks] = contours_value

#Plot whole slice (nucleoplasm and cells with contours)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={'width_ratios': [1, 1]})
vmin_nuc, vmax_nuc = np.percentile(image_chNuclei_selected, (20, 99))

ax1.imshow(contours_image_chNuclei_selected, origin='upper', cmap = 'gray', vmin = vmin_nuc, vmax = vmax_nuc)
ax2.imshow(binary_no_cell_masks)

# Adjust spacing between subplots
plt.tight_layout()

# Generate name for whole slice image
titulo_slice = ("date: " +str(dataset_date) + "\ndataset: " + str(genotype_number) + "\nslice: " + str(analysis_slice) + 
                        "\nbackground: " + str(background) + " au" + "\nsegmentation method: " + segmentation_method)
ax1.set_title(titulo_slice, loc='left')

plt.show()
plt.close(fig) 
file_name_slice_image = (output_folder_images_dataset + "/" + str(dataset_date) + "-" + 
                    str(genotype_number) + "-slice_" + str(analysis_slice) + ".png")
print("Output file:", file_name_slice_image)
fig.savefig(file_name_slice_image, bbox_inches='tight')
 
###Add contours to segmented protein segmented image
###
contours_image_chProtein_segmented = np.copy(image_chProtein_segmented)
contours_image_chProtein_segmented[outline_masks] = contours_value

###Generate images of individual cells
###
iteration = 0
for row in selected_cells_stats:

    fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(20, 5), gridspec_kw={'width_ratios': [1, 1.1, 1, 1]})

    mean_cell_expression_suffix = round(row[3] / row[4])
    file_name_image = (output_folder_images_dataset + "/fluo_" + str(mean_cell_expression_suffix) +
                       "_" + str(dataset_date) + "-" + str(genotype_number) + "-slice_" + str(analysis_slice) + 
                       "-cell_" + str(int(row[0])) + ".png")
    
    #+-450 for SoRa
    #+-200 for Confocal
    plot_y_low = selected_coordinates[iteration][0]
    plot_y_up = selected_coordinates[iteration][1]
    plot_x_low = selected_coordinates[iteration][2]
    plot_x_up = selected_coordinates[iteration][3]
    
    plot_x_low = 0 if plot_x_low < 0 else plot_x_low
    plot_y_low = 0 if plot_y_low < 0 else plot_y_low
    
    #Contrast enhancing for spy650 channel
    ax1.imshow(image_chNuclei_selected, origin='upper', cmap = 'gray', vmin = vmin_nuc, vmax = vmax_nuc)
    ax1.set_xlim(plot_x_low, plot_x_up) #zoom to mask currently analyzed
    ax1.set_ylim(plot_y_up, plot_y_low)
    
    #Add scale bar on ax1 image
    #60.26um x 55.02um field view size (2300 x 2100 pixels). Pixel size is 0.0262 um (26.2 nm).
    #Thus, 382 pixels is equivalent to 10 000 nm 
    scale_color='white'
    scale_units='μm'
    scale_length = 10  # in data coordinates (micrometers)
    pixel2micrometer_ratio = 1 / 0.0262 #ratio of 1 pixel to distance units
    scale_location = (plot_x_low + 50, plot_y_up - 50)  # starting point of the scale bar
    
    rect = Rectangle(scale_location, scale_length * pixel2micrometer_ratio, 16, edgecolor='black', facecolor=scale_color)
    ax1.add_patch(rect)
    ax1.text(scale_location[0] + scale_length * pixel2micrometer_ratio / 2, scale_location[1] + 0.04,
            f'{scale_length} {scale_units}', color=scale_color, ha='center', va='bottom', fontsize = 20)
    ####
    ####
    
    vmin = np.percentile(image_chProtein_selected, 20) #set contrast
    vmax = np.percentile(image_chProtein_selected, 99.8) #set contrast
    im = ax2.imshow(image_chProtein_selected, origin='upper', vmin = vmin, vmax = vmax)
    ax2.set_xlim(plot_x_low, plot_x_up) #zoom to mask currently analyzed
    ax2.set_ylim(plot_y_up, plot_y_low)
    cbar = plt.colorbar(im, ax=ax2, fraction=0.04, pad=0.04) # Add a colorbar to the first subplot
#     ax2.set_yticks([])
#     ax2.set_xticks([])
    
    ax3.set_facecolor('black')
    ax3.imshow(contours_image_chProtein_segmented[plot_y_low:plot_y_up, plot_x_low:plot_x_up], origin='upper', cmap=cmap, interpolation='nearest')
#     ax3.set_yticks([])
#     ax3.set_xticks([])
    
    #Plot nucleoplasm
    ax4.imshow(selected_nucleoplasms[iteration],origin='upper')
#     ax4.set_xlim(plot_x_low, plot_x_up) #zoom to mask currently analyzed
#     ax4.set_ylim(plot_y_up, plot_y_low)
#     ax4.set_yticks([])
#     ax4.set_xticks([])
    
    # Adjust spacing between subplots
    plt.tight_layout()
    titulo = ("date: " +str(dataset_date) + "\ndataset: " + str(genotype_number) + "\nslice: " + str(analysis_slice) + 
                       "\ncell: " + str(int(row[0])) + "\nsegmentation method: " + segmentation_method)
    ax1.set_title(titulo, loc='left')

    # Save the figure as PNG with specified DPI
    print("Output file:", file_name_image)
    fig.savefig(file_name_image, bbox_inches='tight')
    plt.show()
    plt.close(fig) 
    
    iteration = iteration + 1


In [ ]:
## Generate a file for condensates that includes masks 
###Generate a file for masks

#condensate_stats = np.append(condensate_stats, np.array([[condensates_in_cell_mask, 
        #cd_id, cd_xpixel, cd_ypixel, cd_raw_mass, cd_size, cd_ecc, cd_sigma]]), axis=0)

rows2write_condensates = []
rows2write_cells = []

headers_condensates = ("date" + "\t" + "dataset" + "\t" + "cell_Id" +
           "\t" + "condensateID" + "\t" + "x" + "\t" + "y" + 
           "\t" + "raw_mass" + "\t" + "size" + "\t" + "ecc\n")

headers_cells = ("date" + "\t" + "dataset" + "\t" + "z-slice" + "\t" + "cell_Id"
           "\t" + "x" + "\t" + "y" + "\t" + "raw_exp" + "\t" + "cell_area" + "\t" + "background" +
           "\t" + "nuc_intensity" + "\t" + "nuc_area\n")

#print(headers)
rows2write_condensates.append(headers_condensates)
rows2write_cells.append(headers_cells)

#print("condensate_stats", condensate_stats)
for row in condensate_stats:
    if row[0] == 0:
        continue #skip iteration if condensate is not assigned to any cell mask
    if row[0] not in selected_cells_stats[:,0]:
        continue #skip iteration if current condensate has not been assigned to any manually selected cell mask
    
    data_row = (str(dataset_date) + "\t" + str(genotype_number) + 
            "\t" + str(row[0]) + "\t" + str(row[1]) + "\t" + str(row[2]) + 
            "\t" + str(row[3]) + "\t" + str(row[4]) + "\t" + str(row[5]) + 
            "\t" + str(row[6]) + "\n")
        
    rows2write_condensates.append(data_row)


# selected_cells_stats = np.append(selected_cells_stats, np.array([[mask_number, 
#                 center_cell_x, center_cell_y, raw_cell_expression, cell_area, background, 
#                 raw_nucleoplasm_expression, nucleoplasm_area]]), axis=0)
for row in selected_cells_stats:
    
    data_row = (str(dataset_date) + "\t" + str(genotype_number) + "\t" + str(analysis_slice) + 
            "\t" + str(row[0]) + "\t" + str(row[1]) + "\t" + str(row[2]) + 
            "\t" + str(row[3]) + "\t" + str(row[4]) + "\t" + str(row[5]) + 
            "\t" + str(row[6]) + "\t" + str(row[7]) + "\n")
    
    rows2write_cells.append(data_row)
    
#Create folder and files for analysis
# Specify the folder path
if not os.path.exists(output_folder): # Check if the folder exists
    #Create the folder
    os.makedirs(output_folder)
    print("Folder created successfully!")
else:
    print("Folder already exists.")

#Create files with the analysis
try:
    with open(filename_condensates, 'w') as f:
        for row in rows2write_condensates:
            f.write(row)
            
except IOError:
    print("Could not open file {filename_condensates} for writing") #Handle the case where the file cannot be opened or written to


try:
    with open(filename_cells, 'w') as f:
        for row in rows2write_cells:
            f.write(row)
            
except IOError:
    print("Could not open file {filename_cells} for writing") #Handle the case where the file cannot be opened or written to

print("Files:\n", filename_condensates, "\nand\n", filename_cells,"\ngenerated")


In [ ]:
# Generate a file for the parameters used in the current analysis

##Get file name of the script (for Jupyter)
##

##Obtain date
from datetime import datetime
current_date = datetime.now()
formatted_date = current_date.strftime("%Y-%m-%d")

code_date_parameters = 'SoRa1-' + formatted_date + "\n"
date_parameters = "date: " + str(dataset_date) + "\n"
dataset_parameters = "dataset: " + str(genotype_number) + "\n"
z_slice_parameters = "z-slice: " + str(analysis_slice) + "\n"
model_parameters = "cell segmentation model: cellpose-" + str(model_type) + "\n"
segmentation_method_parameters = "condensate segmentation method: " + segmentation_method + "\n"
low_sigma_parameters = "low sigma (pixels): " + str(low_sigma) + "\n"
high_sigma_parameters = "high sigma (pixels): " + str(high_sigma) + "\n"
gaussian_threshold_parameters = "gaussian threshold: " + str(gaussian_threshold) + "\n"
min_size_parameters = "min condensate size (pixels): " + str(min_size) + "\n"
erosion_parameters = "layers eroded around condensates (pixels): " + str(erosion_size) + "\n\n"
Comments_parameters = "Comments: " + Comments + "\n"

print(code_date_parameters)
print(dataset_parameters)
print(z_slice_parameters)
print(model_parameters)
print(segmentation_method_parameters)
print(low_sigma_parameters)
print(high_sigma_parameters)
print(gaussian_threshold_parameters)
print(min_size_parameters)
print(erosion_parameters)
print(Comments_parameters)
# rows2write_parameters = []


if not os.path.exists(output_folder): # Check if the folder exists
    #Create the folder
    os.makedirs(output_folder)
    print("Folder created successfully!")
else:
    print("Folder already exists.")

try:
    with open(filename_parameters, 'w') as f:
        f.write(code_date_parameters)
        f.write(dataset_parameters)
        f.write(z_slice_parameters)
        f.write(model_parameters)
        f.write(segmentation_method_parameters)
        f.write(low_sigma_parameters)
        f.write(high_sigma_parameters)
        f.write(gaussian_threshold_parameters)
        f.write(min_size_parameters)
        f.write(erosion_parameters)
        f.write(Comments_parameters)
            
except IOError:
    print("Could not open file {filename_parameters} for writing") #Handle the case where the file cannot be opened or written to

print("File:\n", filename_parameters, "\ngenerated")
